In [1]:
import json
import numpy as np
from z3 import *
import galois

# Load code 

In [2]:
dict_load = json.load(open('../../Unfolded code search/UnfoldedCode.json', 'r'))

# stabilizers
stabilizers = dict_load['stabilzers']

# X_L rotations
rotations = dict_load['rotations']

# information qubits where X physical = X logical
info_qubits = dict_load['info_qubits']

# out qubits hosting magic state
out_qubits = dict_load['out_qubits']

# Remove merging stabilizer

In [3]:
# remove stabilizer to allow merge with surface code
stabilizers.remove([19, 14, 15, 18])

In [4]:
# convert stabilizers to empty numpy array
max_len = max(map(len, stabilizers))
stabilizers = np.array([sublist + [np.nan] * (max_len - len(sublist)) for sublist in stabilizers])
stabilizers

array([[ 0.,  2., nan, nan],
       [ 1.,  0.,  3.,  4.],
       [ 5.,  1., nan, nan],
       [ 2.,  6., nan, nan],
       [ 3.,  2.,  7.,  8.],
       [ 4.,  3.,  8.,  9.],
       [10.,  4.,  5.,  9.],
       [ 6.,  7., 11., 12.],
       [ 7.,  8., 12., 13.],
       [ 8.,  9., 13., 14.],
       [15.,  9., 10., 14.],
       [11., 16., nan, nan],
       [16., 12., 13., 17.],
       [17., 13., 14., 18.]])

# Surface code stabilizer

In [5]:
# distance against bit-flip
dx = 3
# distance against phase-flip
dz = 13

In [6]:
# qubit in the surface code that will host the magic state
out_qubits = range(int(np.nanmax(stabilizers)) + 1, int(np.nanmax(stabilizers)) + 1 + dx*dz)

In [7]:
# ancilla type (X or Z) of the surface code
def Create_Ancilla_type(dz,dx):
    Ancilla_type = np.full((dz+1,dx+1),' ')

    for i in range(dz+1):
        for j in range(dx+1):
            if (j+i)%2 == 1 and j>0 and j<dx:
                Ancilla_type[i,j] = 'Z'
            if (j+i)%2 == 0 and i>0 and i<dz:
                Ancilla_type[i,j] = 'X'

    return Ancilla_type

In [8]:
Create_Ancilla_type(dx = dx,dz = dz)

array([[' ', 'Z', ' ', ' '],
       [' ', 'X', 'Z', 'X'],
       ['X', 'Z', 'X', ' '],
       [' ', 'X', 'Z', 'X'],
       ['X', 'Z', 'X', ' '],
       [' ', 'X', 'Z', 'X'],
       ['X', 'Z', 'X', ' '],
       [' ', 'X', 'Z', 'X'],
       ['X', 'Z', 'X', ' '],
       [' ', 'X', 'Z', 'X'],
       ['X', 'Z', 'X', ' '],
       [' ', 'X', 'Z', 'X'],
       ['X', 'Z', 'X', ' '],
       [' ', ' ', 'Z', ' ']], dtype='<U1')

In [9]:
# return array of X stabilizers
def Create_X_stabilizers(dz,dx):
    
    Stabilizers_X = np.full(((int(dx/2)+1)*(dz-1),4),np.nan)
    
    Ancilla_type = Create_Ancilla_type(dz,dx)
    Data_number = np.array(range(dx*dz)).reshape((dz,dx))
    
    counter = 0
    
    for i in range(Ancilla_type.shape[0]):
        for j in range(Ancilla_type.shape[1]):

            if Ancilla_type[i,j] == "X":
                if i>0 and j>0:
                    Stabilizers_X[counter][0] = Data_number[i-1,j-1]
                if i<dz and j>0:
                    Stabilizers_X[counter][1] = Data_number[i,j-1]
                if i>0 and j<dx:
                    Stabilizers_X[counter][2] = Data_number[i-1,j]
                if i<dz and j<dx:
                    Stabilizers_X[counter][3] = Data_number[i,j]
                counter += 1
            
    return Stabilizers_X

In [10]:
Create_X_stabilizers(dx = dx,dz = dz)

array([[ 0.,  3.,  1.,  4.],
       [ 2.,  5., nan, nan],
       [nan, nan,  3.,  6.],
       [ 4.,  7.,  5.,  8.],
       [ 6.,  9.,  7., 10.],
       [ 8., 11., nan, nan],
       [nan, nan,  9., 12.],
       [10., 13., 11., 14.],
       [12., 15., 13., 16.],
       [14., 17., nan, nan],
       [nan, nan, 15., 18.],
       [16., 19., 17., 20.],
       [18., 21., 19., 22.],
       [20., 23., nan, nan],
       [nan, nan, 21., 24.],
       [22., 25., 23., 26.],
       [24., 27., 25., 28.],
       [26., 29., nan, nan],
       [nan, nan, 27., 30.],
       [28., 31., 29., 32.],
       [30., 33., 31., 34.],
       [32., 35., nan, nan],
       [nan, nan, 33., 36.],
       [34., 37., 35., 38.]])

In [13]:
# return array of Z stabilizers
def Create_Z_stabilizers(dz,dx):
    
    Stabilizers_Z = np.full((int(dx/2)*(dz+1),4),np.nan)
    
    Ancilla_type = Create_Ancilla_type(dz,dx)
    Data_number = np.array(range(dx*dz)).reshape((dz,dx))
    
    counter = 0
    
    for i in range(Ancilla_type.shape[0]):
        for j in range(Ancilla_type.shape[1]):

            if Ancilla_type[i,j] == "Z":
                if i>0 and j>0:
                    Stabilizers_Z[counter][0] = Data_number[i-1,j-1]
                if i>0 and j<dx:
                    Stabilizers_Z[counter][1] = Data_number[i-1,j]
                if i<dz and j>0:
                    Stabilizers_Z[counter][2] = Data_number[i,j-1]
                if i<dz and j<dx:
                    Stabilizers_Z[counter][3] = Data_number[i,j]
                counter += 1
            
    return Stabilizers_Z

In [14]:
Create_Z_stabilizers(dx = dx,dz = dz)

array([[nan, nan,  0.,  1.],
       [ 1.,  2.,  4.,  5.],
       [ 3.,  4.,  6.,  7.],
       [ 7.,  8., 10., 11.],
       [ 9., 10., 12., 13.],
       [13., 14., 16., 17.],
       [15., 16., 18., 19.],
       [19., 20., 22., 23.],
       [21., 22., 24., 25.],
       [25., 26., 28., 29.],
       [27., 28., 30., 31.],
       [31., 32., 34., 35.],
       [33., 34., 36., 37.],
       [37., 38., nan, nan]])

# Merging stabilizers for lattice surgery

In [24]:
# divide weight-6 measurement into weight-4 and weight-2
merging_stabilizers = np.array([[18,np.nanmax(stabilizers)+1,np.nan,np.nan],
                                [14,15,np.nanmax(stabilizers)+2,np.nanmax(stabilizers)+3]])
merging_stabilizers

array([[18., 19., nan, nan],
       [14., 15., 20., 21.]])

In [25]:
# change indices of surface qubits to be last
X_stab = Create_X_stabilizers(dx = dx,dz = dz) + int(np.nanmax(stabilizers)) + 1
Z_stab = Create_Z_stabilizers(dx = dx,dz = dz) + int(np.nanmax(stabilizers)) + 1

# Parity Check matrix

In [28]:
# Parity check matrix in block form:
# [ Hx ]
# [ Hz ]
# where Hx and Hz are stacked vertically
H = np.zeros((len(stabilizers) + len(merging_stabilizers) + len(X_stab) + len(Z_stab),
              int(np.nanmax(Z_stab))+1),dtype = int)

# update H based on each stabilizer list
def update_H(stabilizer_list, offset):
    for i, row in enumerate(stabilizer_list):
        for value in row:
            if not np.isnan(value):
                H[i + offset][int(value)] = 1

# Add each stabilizer type to H
update_H(stabilizers, 0)
update_H(merging_stabilizers, len(stabilizers))
update_H(X_stab, len(stabilizers) + len(merging_stabilizers))
update_H(Z_stab, len(stabilizers) + len(merging_stabilizers) + len(X_stab))
        
np.set_printoptions(linewidth=np.inf,threshold=np.inf)
print(H)

[[1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 

# Order of the CNOTs

## extend parity check matrix

In [30]:
# list of stabilizers that require a flag qubit
flagged_stab = []
for i in range(len(H)):
    if i<len(stabilizers) and ~np.any(np.isnan(stabilizers[i])):
        flagged_stab.append(i)
    elif len(stabilizers)<=i<len(stabilizers) + len(merging_stabilizers) and ~np.any(np.isnan(merging_stabilizers[i-len(stabilizers)])):
        flagged_stab.append(i)
        
# extend H with one column for measurement, preparation, flag first CNOT and flag second CNOT
H_extended = np.hstack((H,np.ones((H.shape[0],2),dtype = int))) 
flag_matrix = np.zeros((H.shape[0],2),dtype = int)
flag_matrix[flagged_stab] = 1
H_extended = np.hstack((H_extended,flag_matrix)) 

# indices corresponding to measurement, preparation, flag first CNOT and flag second CNOT
prep_index = H_extended.shape[1]-4
measure_index = H_extended.shape[1]-3
flagup_index = H_extended.shape[1]-2
flagdown_index = H_extended.shape[1]-1

## SAT variables

In [31]:
# TimeSteps[i,j,k]=1 -> operation of stabilizer j on qubit k at timestep i 
TimeStep = np.empty((15,H_extended.shape[0], H_extended.shape[1]), dtype=object)

# minimal number of timesteps for the search
assert TimeStep.shape[0] >= 12

# fill TimeSteps with bool variables
for step_index in range(12):
    for j in range(H_extended.shape[1]):
        # unfolded code stabilizers
        for i in range(len(stabilizers) + len(merging_stabilizers)):
            TimeStep[step_index,i, j] = Bool(f'b_{step_index}_{i}_{j}')
            
        # surface code X stabilizers
        for i in range(len(X_stab)):
            stab_index = len(stabilizers) + len(merging_stabilizers) + i
            # CNOT
            if (step_index-1)%6 in range(4) and j == X_stab[i][(step_index-1)%6]:
                TimeStep[step_index,stab_index, j] = BoolVal(True)
            # measurement
            elif (step_index-1)%6 == 4:
                TimeStep[step_index,stab_index, -3] = BoolVal(True)
                for k in range(H_extended.shape[1]):
                    if k != H_extended.shape[1]-3:
                        TimeStep[step_index,stab_index, k] = BoolVal(False)
            # preparation
            elif (step_index-1)%6 == 5:
                TimeStep[step_index,stab_index, -4] = BoolVal(True)
                for k in range(H_extended.shape[1]):
                    if k != H_extended.shape[1]-4:
                        TimeStep[step_index,stab_index, k] = BoolVal(False)
            else:
                TimeStep[step_index,stab_index, j] = BoolVal(False)
            
        # surface code Z stabilizers
        for i in range(len(Z_stab)):
            stab_index = len(stabilizers) + len(merging_stabilizers) + len(X_stab) + i
            # CNOT
            if (step_index-1)%6 in range(4) and j == Z_stab[i][(step_index-1)%6]:
                TimeStep[step_index,stab_index, j] = BoolVal(True)
            # measurement
            elif (step_index-1)%6 == 4:
                TimeStep[step_index,stab_index, -3] = BoolVal(True)
                for k in range(H_extended.shape[1]):
                    if k != H_extended.shape[1]-3:
                        TimeStep[step_index,stab_index, k] = BoolVal(False)
            # preparation
            elif (step_index-1)%6 == 5:
                TimeStep[step_index,stab_index, -4] = BoolVal(True)
                for k in range(H_extended.shape[1]):
                    if k != H_extended.shape[1]-4:
                        TimeStep[step_index,stab_index, k] = BoolVal(False)
            else:
                TimeStep[step_index,stab_index, j] = BoolVal(False)
                
# final stabilizers timesteps in the unfolded code
for step_index in range(12,TimeStep.shape[0]):
    for i in range(len(stabilizers) + len(merging_stabilizers)):
        for j in range(TimeStep.shape[2]):
            TimeStep[step_index,i, j] = Bool(f'b_{step_index}_{i}_{j}')
    # no surface code stabilizer measurement
    for i in range(len(stabilizers) + len(merging_stabilizers),TimeStep.shape[1]):
        for j in range(TimeStep.shape[2]):
            TimeStep[step_index,i, j] = BoolVal(False)
            
# fix preparation and measurement of weight-2 stabilizers of surface code
for step_index in range(TimeStep.shape[0]):
    for i in range(len(stabilizers) + len(merging_stabilizers),TimeStep.shape[1]):
        # preparation
        if step_index < TimeStep.shape[0]-1:
            if is_true(TimeStep[step_index,i,prep_index]) and set([is_true(TimeStep[step_index+1,i,j]) for j in range(TimeStep.shape[2])]) == {False}:
                TimeStep[step_index,i,prep_index] = BoolVal(False)
                TimeStep[step_index+2,i,prep_index] = BoolVal(True)
        # measurement
        if step_index > 0:
            if is_true(TimeStep[step_index,i,measure_index]) and set([is_true(TimeStep[step_index-1,i,j]) for j in range(TimeStep.shape[2])]) == {False}:
                TimeStep[step_index,i,measure_index] = BoolVal(False)
                TimeStep[step_index-2,i,measure_index] = BoolVal(True)

## SAT constraints

In [32]:
s = SolverFor("QF_FD")

### sum of each timestep equal to H

for i in range(len(stabilizers) + len(merging_stabilizers)):
    for j in range(TimeStep.shape[2]):
        constraint = AtLeast(*[TimeStep[step_index, i, j] for step_index in range(TimeStep.shape[0])], int(2*H_extended[i][j]))
        s.add(constraint)
        constraint = AtMost(*[TimeStep[step_index, i, j] for step_index in range(TimeStep.shape[0])], int(2*H_extended[i][j]))
        s.add(constraint)

### qubit and ancilla used once in each timestep

for step_index in range(TimeStep.shape[0]):
    for i in range(len(stabilizers) + len(merging_stabilizers)):
        # unflagged stabilizer
        if i not in flagged_stab:
            constraint = AtMost(*TimeStep[step_index,i,:], 1)
            s.add(constraint)
        # flagged stabilizer
        else:
            constraint = AtMost(*TimeStep[step_index,i,:], 2)
            s.add(constraint)
            constraint_prep = Implies(TimeStep[step_index,i,prep_index],AtMost(*TimeStep[step_index,i,:], 1))
            constraint_measure = Implies(TimeStep[step_index,i,measure_index],AtMost(*TimeStep[step_index,i,:], 1))
            constraint_flagup = Implies(TimeStep[step_index,i,flagup_index],AtMost(*TimeStep[step_index,i,:], 1))
            constraint_flagdown = Implies(TimeStep[step_index,i,flagdown_index],AtMost(*TimeStep[step_index,i,:], 1))
            s.add(constraint_prep,constraint_measure,constraint_flagup,constraint_flagdown)
        
    # only one 1 in a column
    for i in range(H.shape[1]):
        constraint = AtMost(*TimeStep[step_index,:,i], 1)
        s.add(constraint)

### preparation before CNOT

for i in range(len(stabilizers) + len(merging_stabilizers)):
    for step_index in range(1,TimeStep.shape[0]):
        prep = TimeStep[step_index,i,prep_index]
        one_occurence = And([
                            And(AtMost(*TimeStep[:step_index,i,j],1),AtLeast(*TimeStep[:step_index,i,j],1))
                            for j in range(TimeStep.shape[2]) if H_extended[i,j] == 1
                            ])
        zero_occurence = And([
                            AtMost(*TimeStep[:step_index,i,j],0) 
                            for j in range(TimeStep.shape[2]) if H_extended[i,j] == 1
                            ])
        constraint = Or(Implies(prep,one_occurence),Implies(prep,zero_occurence))
        s.add(constraint)     
        
### measurement after CNOT

for i in range(len(stabilizers) + len(merging_stabilizers)):
    for step_index in range(TimeStep.shape[0]-1):
        measure = TimeStep[step_index,i,measure_index]
        one_occurence = And([
                            And(AtMost(*TimeStep[step_index+1:,i,j],1),AtLeast(*TimeStep[step_index+1:,i,j],1))
                            for j in range(TimeStep.shape[2]) if H_extended[i,j] == 1
                            ])
        zero_occurence = And([
                            AtMost(*TimeStep[step_index+1:,i,j],0) 
                            for j in range(TimeStep.shape[2])
                            ])
        constraint = Or(Implies(measure,one_occurence),Implies(measure,zero_occurence))
        s.add(constraint)  

### first CNOT of flag qubit before second CNOT

for i in range(len(stabilizers) + len(merging_stabilizers)):
    for step_index in range(1,TimeStep.shape[0]):
        constraint = Implies(
            TimeStep[step_index,i,flagdown_index],
            AtLeast(*TimeStep[:step_index,i,flagup_index],1)
        )
        s.add(constraint)
                
### second CNOT of flag qubit after first CNOT

for i in range(len(stabilizers) + len(merging_stabilizers)):
    for step_index in range(TimeStep.shape[0]-1):
        constraint = Implies(
            TimeStep[step_index,i,flagup_index],
            AtLeast(*TimeStep[step_index:,i,flagdown_index],1)
        )
        s.add(constraint)

### first CNOT of flag qubit before double CNOT

for i in range(TimeStep.shape[1]):
    for step_index in range(1,TimeStep.shape[0]):
        double_cnot = AtLeast(*TimeStep[step_index,i],2)
        one_flag_before = And(
                            And(AtLeast(*TimeStep[:step_index,i,flagup_index],1),AtMost(*TimeStep[:step_index,i,flagup_index],1)),
                            AtMost(*TimeStep[:step_index,i,flagdown_index],0)
                            )
        two_flags_before = And(
                            And(AtLeast(*TimeStep[:step_index,i,flagup_index],2),AtMost(*TimeStep[:step_index,i,flagup_index],2)),
                            And(AtLeast(*TimeStep[:step_index,i,flagdown_index],1),AtMost(*TimeStep[:step_index,i,flagdown_index],1))
                            )
        constraint = Or(Implies(double_cnot,one_flag_before),Implies(double_cnot,two_flags_before))
        s.add(constraint)
        

### no single CNOT in a row at the end of flagged stabilizer (for distance against bit-flip)

for i in flagged_stab:
    for step_index in range(TimeStep.shape[0]-1):
        flag_down = TimeStep[step_index,i,flagdown_index]
        prep = Or([TimeStep[step_index_2,i,prep_index] for step_index_2 in range(step_index,TimeStep.shape[0])])
        two_single_cnot = AtLeast(*[TimeStep[step_index_2,i,j] 
                                   for j in range(0,TimeStep.shape[2]-4) 
                                   for step_index_2 in range(step_index,TimeStep.shape[0])]
                                  ,2)
        constraint = Or(Implies(flag_down,prep),Implies(flag_down,Not(two_single_cnot)))
        s.add(constraint)
        
### no single CNOT in a row at the beginning of flagged stabilizer (for distance against bit-flip)

for i in flagged_stab:
    for step_index in range(1,TimeStep.shape[0]):
        flag_up = TimeStep[step_index,i,flagdown_index]
        two_single_cnot = AtLeast(*[
                                    And(AtLeast(*[TimeStep[step_index_2,i,j] for j in range(TimeStep.shape[2]-4)],1),
                                        AtMost(*[TimeStep[step_index_2,i,j] for j in range(TimeStep.shape[2]-4)],1))
                                    for step_index_2 in range(step_index)
                                  ],2)
        constraint = Implies(flag_up,Not(two_single_cnot))
        s.add(constraint)

### first round of stabilizer measurement in four timestep

for i in flagged_stab:
    for step_index in range(TimeStep.shape[0]):
        prep = TimeStep[step_index,i,prep_index]
        constraint = And([
                        Implies(TimeStep[step_index_2,i,prep_index],TimeStep[step_index_2+5,i,measure_index])
                        for step_index_2 in range(step_index-5)
                        ])
        s.add(Implies(prep,constraint))

### Measure weight-2 stabilizers in 2 timesteps

for i in range(len(stabilizers) + len(merging_stabilizers)):
    if i not in flagged_stab:
         for step_index in range(TimeStep.shape[0]-3):  
            constraint = Implies(TimeStep[step_index,i,prep_index],TimeStep[step_index+3,i,measure_index])
            s.add(constraint)
            
### Limit position of the last first measurement

for i in range(len(stabilizers) + len(merging_stabilizers)):
    for step_index in range(8,TimeStep.shape[0]-1):
        measure = TimeStep[step_index,i,measure_index]
        first_measurement = AtLeast(*[TimeStep[step_index_2,i,measure_index] for step_index_2 in range(step_index+1,TimeStep.shape[0])],1)
        constraint = And(measure,first_measurement)
        s.add(Not(constraint))
        
### CNOT gate just before X^1/4 rotation

for rotation in rotations:

    LastRound = BoolVal(False)
    for i in range(TimeStep.shape[1]):
        LastRound = Or(LastRound,TimeStep[TimeStep.shape[0]-2,i,rotation] == True)
    
    s.add(LastRound == True)

## solve

In [33]:
%%time

if s.check() == sat:
    print('sat')
    m = s.model()
else:
    print("unsat")

sat
CPU times: total: 7.05 s
Wall time: 7.14 s


## retriev result

In [34]:
# Create a numpy array to store the result
TimeStep_result = np.zeros((TimeStep.shape[0], TimeStep.shape[1], TimeStep.shape[2]), dtype=int)
    
# Store the Boolean values in the numpy array
for i in range(TimeStep.shape[0]):
    for j in range(TimeStep.shape[1]):
        for k in range(TimeStep.shape[2]):
            if j < len(stabilizers) + len(merging_stabilizers):
                TimeStep_result[i, j, k] = is_true(m[TimeStep[i, j, k]])
            else:
                TimeStep_result[i, j, k] = is_true(TimeStep[i, j, k])

## print result

In [35]:
result = []

for i in range(H_extended.shape[0]):
    result.append([])
    for step_index in range(TimeStep.shape[0]):
        result[i].append(list(np.where(TimeStep_result[step_index,i])[0].tolist()))
        
for i in range(len(result)):
    for j in range(len(result[i])):
        for k in range(len(result[i][j])):
            if result[i][j][k] == prep_index:
                result[i][j][k] = 'P'
            if result[i][j][k] == measure_index:
                result[i][j][k] = 'M'
            if result[i][j][k] == flagup_index:
                result[i][j][k] = 'Fup'
            if result[i][j][k] == flagdown_index:
                result[i][j][k] = 'Fdo'

In [36]:
# ensure columns are aligned in the print
column_widths = [
    max(len(str(item)) if isinstance(item, list) else len(f"{item:.5f}") for item in col) 
    for col in zip(*result)
]

for row in result:
    print(" ".join(
        (str(value) if isinstance(value, list) else f"{value}").rjust(width) 
        for value, width in zip(row, column_widths)
    ))

   []      []       []       []    ['P']      [2]     [0]   ['M']      []      []       []    ['P']     [2]  [0] ['M']
['P'] ['Fup']   [1, 4]   [0, 3]  ['Fdo']    ['M']   ['P'] ['Fup']  [0, 3]      []       []      [1] ['Fdo']  [4] ['M']
['P']     [1]      [5]    ['M']       []       []      []      []      []      []       []    ['P']     [5]  [1] ['M']
   []   ['P']      [2]      [6]    ['M']       []   ['P']     [2]     [6]   ['M']       []       []      []   []    []
['P'] ['Fup']   [3, 7]   [2, 8]  ['Fdo']    ['M']      []   ['P'] ['Fup']  [2, 8]       []      [7] ['Fdo']  [3] ['M']
   []      []    ['P']  ['Fup']   [8, 9]   [3, 4] ['Fdo']   ['M']   ['P'] ['Fup']      [4]   [3, 9] ['Fdo']  [8] ['M']
   []   ['P']  ['Fup']  [9, 10]   [4, 5]  ['Fdo']   ['M']   ['P'] ['Fup']  [5, 9]       []      [4] ['Fdo'] [10] ['M']
   []   ['P']  ['Fup'] [11, 12]   [6, 7]  ['Fdo']   ['M']   ['P']     [7] ['Fup']  [6, 11]       [] ['Fdo'] [12] ['M']
['P'] ['Fup']  [8, 12]  [7, 13]  ['Fdo']    ['M'

# Logical qubits

In [37]:
GF2 = galois.GF2
Hx = GF2(H[:len(stabilizers)+len(X_stab)+len(merging_stabilizers)])
# vectors in the kernel of X stabilizer matrix
Gz = Hx.null_space()
Gz = np.array(Gz)
# remove Z stabilizers
Gz = Gz[:5]

Gz

array([[1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1],
       [0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1],
       [0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1]], dtype=uint8)

In [38]:
# separate Z_L of surface code
Gz[0] = (Gz[0] + Gz[4])%2
Gz[1] = (Gz[1] + Gz[4])%2
Gz

array([[1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1]], dtype=uint8)

In [39]:
# rearrange by multiplying by Z stabilizers
Hz = H[len(stabilizers)+len(X_stab)+len(merging_stabilizers):]
Gz[0] = (Gz[0] + Hz[1])%2
Gz[2] = (Gz[2] + Hz[1])%2
Gz[4] = (Gz[4] + Hz[0] + Hz[1])%2
Gz

array([[1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1]], dtype=uint8)

In [40]:
Logical_qubits = [np.where(Gz[i])[0].tolist() for i in range(Gz.shape[0])]
for row in Logical_qubits:
    print(row)

[0, 2, 4, 6, 8, 10, 11, 13, 15, 16, 18, 19, 20]
[1, 4, 5, 9, 10, 11, 12, 13, 16, 17]
[3, 4, 8, 9, 13, 14, 17, 18, 19, 20]
[7, 8, 9, 10, 12, 13, 14, 15]
[11, 12, 13, 14, 15, 16, 17, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45, 48, 51, 54, 57]


# Add info qubits for surface code

In [41]:
info_qubits += [out_qubits[0]+dx-1+j + dx*2*i for i in range(int((dz+1)/2)) for j in range(2)][:-1]

In [42]:
print(info_qubits)

[9, 12, 13, 14, 18, 21, 22, 27, 28, 33, 34, 39, 40, 45, 46, 51, 52, 57]


# save stabilizer

In [43]:
stabilizer_type = ['X']*len(stabilizers) + ['X']*len(merging_stabilizers) + ['X']*len(X_stab)  + ['Z']*len(Z_stab)

In [45]:
dict_save = {}
dict_save['stabilzers'] = result
dict_save['rotations'] = rotations
dict_save['info_qubits'] = info_qubits
dict_save['out_qubits'] = list(out_qubits)
dict_save['stabilizer_type'] = stabilizer_type
dict_save['logical_qubits'] = Logical_qubits
dict_save['dx'] = dx
dict_save['dz'] = dz

with open('UnfoldedCode.json', 'w') as json_file:
    json.dump(dict_save, json_file)